# A2 — SegFormer-B0: S0 và S1 (train / resume / test)

Chọn **Runtime → Change runtime type → GPU** rồi **Run all**. Chỉ cần file notebook này.
Mặc định chạy lần lượt **hai run** seed 69420: **S0** tắt augmentation màu, **S1** bật augmentation màu.
Mỗi run train tối đa 40 epochs, patience 8, batch 4, ảnh 512×512; AdamW lr **0.00006**, weight decay 0.01,
CE + foreground Dice, FP32. Hai cấu hình được pin trước train/test; pilot xác nhận khả năng chạy GPU.

Khởi tạo cùng SegFormer-B0 pretrained ADE20K, thay classifier 17 lớp, **full fine-tune toàn bộ tham số**.
Không dùng checkpoint U-Net để khởi tạo. Dataset/cache và seed-36 split giống U-Net.
Proposal đã được duyệt theo xác nhận của Khải ngày 26/10. Mỗi run có pilot riêng, pilot model bị bỏ trước main.

Kết quả: `MyDrive/dlbench_a2_segformer/S_seed69420_v1/S0/` và `S1/`.
Checkpoint lưu mỗi epoch; mở lại cùng notebook/settings để tiếp tục epoch đã hoàn tất.
Chỉ mở một notebook ghi vào cùng PAIR_TAG. Không đảm bảo một phiên Colab chạy xong cả hai run.


In [ ]:
# 1. Settings — giữ nguyên khi resume; nếu thay settings hãy dùng PAIR_TAG mới.
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
from pathlib import Path
PAIR_TAG = 'S_seed69420_v1'
RUNS = ['S0', 'S1']           # ['S0'] hoặc ['S1'] nếu muốn chạy từng run ở các phiên khác nhau
RUN_MAIN = True
RUN_TEST_AFTER_TRAIN = True
BATCH_SIZE = 4
MAX_EPOCHS = 40
PATIENCE = 8
LEARNING_RATE = 0.00006
PILOT_STEPS = 20
VERIFY_ALL_MASKS = True
DOWNLOAD_RESULTS = False     # Drive luôn có ZIP; True để tải ZIP từng run xuống máy
CODE_REVISION = '7df323698d1ae4d5122850a54eb38b01609d80f1'
REPO_URL = 'https://github.com/TuanKhai1210/deep-learning-benchmark-suite.git'
assert PAIR_TAG and all(c.isalnum() or c in '_-' for c in PAIR_TAG)
assert RUNS and len(set(RUNS)) == len(RUNS) and set(RUNS) <= {'S0', 'S1'}
assert type(PILOT_STEPS) is int and PILOT_STEPS >= 2


## 2. Drive và source đã pin

Giữ torch/torchvision của Colab; cài Pillow 12.3.0 và Transformers 4.57.6 như source A2.
Chạy trên GPU (ưu tiên cùng T4 như U-Net để so sánh chi phí trong cùng loại thiết bị).
Nếu thư viện đã import với phiên bản khác, restart session rồi Run all.
Deterministic algorithms dùng warn_only=True cho CUDA bilinear backward, được ghi trong config;
không hứa GPU tái lập giống từng bit. Model pretrained public được tải từ Hugging Face, không cần token.


In [ ]:
import subprocess, sys
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/dlbench_a2_segformer_code')
if not REPO.exists():
    subprocess.check_call(['git', 'clone', '--no-checkout', REPO_URL, str(REPO)])
else:
    assert (REPO / '.git').is_dir()
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain']).strip(), 'Source đã sửa: giữ nguyên để kiểm tra'
subprocess.check_call(['git', '-C', str(REPO), 'checkout', '--detach', CODE_REVISION])
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD']).decode().strip() == CODE_REVISION
already_loaded = {name: getattr(sys.modules.get(module), '__version__', None)
                  for name, module in [('Pillow', 'PIL'), ('Transformers', 'transformers')]}
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO) + '[a2]',
                       'gdown==5.2.0', 'google-api-python-client', 'tqdm'])
for name, expected in [('Pillow', '12.3.0'), ('Transformers', '4.57.6')]:
    if already_loaded[name] and already_loaded[name] != expected:
        raise RuntimeError(f'{name} đã import phiên bản khác. Restart session rồi Run all.')
sys.path.insert(0, str(REPO / 'src'))
import torch, PIL, transformers
torch.set_num_threads(2)
torch.set_default_dtype(torch.float32)
assert PIL.__version__ == '12.3.0' and transformers.__version__ == '4.57.6'
assert torch.cuda.is_available(), 'Chọn GPU runtime trước khi tiếp tục'
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.allow_tf32 = False
torch.backends.cuda.matmul.allow_tf32 = False
os.chdir(REPO)
DRIVE = Path('/content/drive/MyDrive')
PAIR_ROOT = DRIVE / 'dlbench_a2_segformer' / PAIR_TAG
CACHE = DRIVE / 'dlbench_a2_cache'
PAIR_ROOT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), '| torch:', torch.__version__)


## 3. Helper và manifest tích hợp

Các helper data/pilot, reconnect recovery và test được dùng lại từ notebook U-Net đã kiểm tra.
Bundle hash khóa toàn bộ helper cho cặp run này. Không thay mã helper giữa chừng khi resume.


In [ ]:
#@title Embedded helpers / manifest — tự động, không cần sửa
import base64, hashlib, importlib.util, json, zlib
HELPERS = {'a2_support': '"""Colab orchestration around pinned A2 modules; no second epoch trainer.\n\nThis helper is embedded in the notebook. Dataset and main training are performed\nonly when the user executes that notebook, never at helper import.\n"""\nfrom copy import deepcopy\nfrom datetime import datetime, timezone\nfrom pathlib import Path, PurePosixPath\nimport gc\nimport hashlib\nimport json\nimport os\nimport shutil\nimport stat\nimport time\nimport uuid\nimport zipfile\n\nCODE_REVISION = "7df323698d1ae4d5122850a54eb38b01609d80f1"\nDATA_FILE_ID = "1a7C5WOkM5tUqsov0pIaYKTOjp0b5AitN"\nDATA_BYTES = 553029970\nDATA_SHA256 = "caf93d992b85343cfc500386c94ca9ffa6c63e2284e04127753a40db8f33ba4c"\nAPPROVAL = "User confirmed instructor approval on 2026-10-26 (Asia/Bangkok)"\nCUDA_POLICY = {"deterministic_algorithms": True, "warn_only": True,\n               "reason": "CUDA bilinear interpolation backward has no strict deterministic implementation"}\n\n\ndef prepare_base_config(config):\n    config = deepcopy(config)\n    if config["model"]["name"] != "unet":\n        raise ValueError("This setup is specifically for U-Net")\n    config["protocol"]["cuda_determinism"] = deepcopy(CUDA_POLICY)\n    return config\n\n\ndef seed_for_unet(seed, *, deterministic=True):\n    """Keep shared seeds; allow documented CUDA interpolation warnings explicitly."""\n    import torch\n    from dlbench.common.reproducibility import seed_everything\n    seed_everything(seed, deterministic=deterministic)\n    torch.use_deterministic_algorithms(deterministic, warn_only=deterministic)\n\n\ndef install_runtime_policy(config):\n    import torch\n    import dlbench.a2.trainer as trainer\n    if config["protocol"].get("cuda_determinism") != CUDA_POLICY:\n        raise ValueError("Declare the U-Net CUDA warning policy in the effective config")\n    torch.set_default_dtype(torch.float32)\n    # The core trainer resets seeds internally; this explicit adapter survives\n    # that reset. Effective config and helper snapshot/hash record the policy.\n    trainer.seed_everything = seed_for_unet\n\n\ndef sha256_file(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                    ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)\n    temporary.write_text(json.dumps(value, indent=2, ensure_ascii=False, allow_nan=False) + "\\n",\n                         encoding="utf-8")\n    os.replace(temporary, path)\n\n\ndef acquire_archive(path, downloader, *, expected_sha=DATA_SHA256, expected_size=DATA_BYTES):\n    """Verified cache; failed downloads never replace the authoritative archive."""\n    path = Path(path)\n    if path.exists():\n        if path.stat().st_size != expected_size or sha256_file(path) != expected_sha:\n            raise ValueError("Existing archive differs from expected EDA source; preserve and inspect it")\n        return path\n    path.parent.mkdir(parents=True, exist_ok=True)\n    temporary = path.with_name(path.name + ".partial-" + uuid.uuid4().hex)\n    downloader(temporary)\n    if temporary.stat().st_size != expected_size or sha256_file(temporary) != expected_sha:\n        raise ValueError(f"Downloaded bytes differ from EDA source; partial retained: {temporary}")\n    os.replace(temporary, path)\n    return path\n\n\ndef download_from_drive(destination):\n    """Try public Drive download, then authenticated read-only Drive API in Colab."""\n    import gdown\n    try:\n        result = gdown.download(id=DATA_FILE_ID, output=str(destination), quiet=False)\n        if result and Path(destination).stat().st_size == DATA_BYTES:\n            return\n    except Exception as error:\n        print("Public Drive download unavailable:", type(error).__name__)\n    print("Using authenticated Drive read access; choose the account that can open the team folder.")\n    from google.colab import auth\n    import google.auth\n    from googleapiclient.discovery import build\n    from googleapiclient.http import MediaIoBaseDownload\n    auth.authenticate_user()\n    credentials, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])\n    service = build("drive", "v3", credentials=credentials, cache_discovery=False)\n    request = service.files().get_media(fileId=DATA_FILE_ID, supportsAllDrives=True)\n    with Path(destination).open("wb") as stream:\n        download = MediaIoBaseDownload(stream, request, chunksize=16 * 1024 * 1024)\n        done = False\n        while not done:\n            progress, done = download.next_chunk(num_retries=3)\n            if progress:\n                print(f"Dataset download: {progress.progress():.0%}")\n\n\ndef extract_material(archive_path, destination, *, expected_sha=DATA_SHA256):\n    """Extract only material_version + license/readme; preserve original source paths."""\n    archive_path, destination = Path(archive_path), Path(destination)\n    if sha256_file(archive_path) != expected_sha:\n        raise ValueError("Archive hash mismatch before extraction")\n    marker = destination / "extraction.json"\n    if destination.exists():\n        if not marker.is_file():\n            raise ValueError("Incomplete local extraction exists; use a fresh extraction directory")\n        record = json.loads(marker.read_text(encoding="utf-8"))\n        if record["archive_sha256"] != expected_sha:\n            raise ValueError("Extraction cache identity mismatch")\n        root = destination / record["material_root"]\n        if not (root / "instances_val_trashcan.json").is_file():\n            raise ValueError("Extraction cache is incomplete")\n        return root\n    temporary = destination.with_name(destination.name + ".partial-" + uuid.uuid4().hex)\n    temporary.mkdir(parents=True, exist_ok=False)\n    with zipfile.ZipFile(archive_path) as archive:\n        candidates = []\n        for info in archive.infolist():\n            name = info.filename.replace("\\\\", "/")\n            parts = PurePosixPath(name)\n            if parts.is_absolute() or ".." in parts.parts or any(":" in p for p in parts.parts):\n                raise ValueError("Unsafe archive member")\n            if stat.S_IFMT(info.external_attr >> 16) == stat.S_IFLNK:\n                raise ValueError("Archive symlinks are unsupported")\n            if parts.name == "instances_train_trashcan.json" and "material_version" in parts.parts:\n                candidates.append(parts.parent)\n        if len(candidates) != 1:\n            raise ValueError("Expected exactly one material_version annotation directory")\n        prefix = candidates[0]\n        extracted = 0\n        for info in archive.infolist():\n            path = PurePosixPath(info.filename.replace("\\\\", "/"))\n            selected = path.is_relative_to(prefix) or path.name.lower().startswith(("license", "readme"))\n            if not selected or info.is_dir():\n                continue\n            target = temporary / str(path)\n            target.parent.mkdir(parents=True, exist_ok=True)\n            with archive.open(info) as source, target.open("wb") as output:\n                shutil.copyfileobj(source, output)  # ZIP CRC is checked on complete read.\n            extracted += 1\n    if not (temporary / str(prefix) / "instances_val_trashcan.json").is_file():\n        raise ValueError("Missing material validation annotations")\n    write_json(temporary / "extraction.json", {"archive_sha256": expected_sha,\n               "material_root": str(prefix), "files": extracted})\n    os.replace(temporary, destination)\n    return destination / str(prefix)\n\n\ndef make_datasets(root, manifest, config):\n    from dlbench.a2.data import TrashCanDataset\n    args = dict(seed=config["run"]["seed"], color_jitter=False)\n    train = TrashCanDataset(root, manifest, "train", config["preprocessing"], **args)\n    validation = TrashCanDataset(root, manifest, "validation", config["preprocessing"], **args)\n    return train, validation\n\n\ndef verify_training_data(train, validation):\n    """All train/validation native masks and RGB dimensions, with no test evaluation."""\n    from PIL import Image\n    from dlbench.a2.converter import rasterize_semantic\n    from tqdm.auto import tqdm\n    checked = {}\n    for dataset in (train, validation):\n        for row in tqdm(dataset.rows, desc=f"Verify {dataset.split}"):\n            record = dataset.records[row["image_id"]]\n            with Image.open(dataset.root / row["relative_path"]) as image:\n                if image.size != (record["width"], record["height"]):\n                    raise ValueError(f"RGB dimension mismatch: {row[\'image_id\']}")\n                image.verify()\n            target = rasterize_semantic(record)[0]\n            if hashlib.sha256(target.tobytes()).hexdigest() != row["target_sha256"]:\n                raise ValueError(f"Native target mismatch: {row[\'image_id\']}")\n        checked[dataset.split] = len(dataset)\n    return {"checked": checked, "manifest_sha256": train.manifest_digest,\n            "test_evaluated": False}\n\n\ndef pilot(config, dataset, *, steps=20, device="cuda", require_cuda=True):\n    """Fixed real batch diagnostic and train-step/VRAM pilot, discarded before main."""\n    import torch\n    from dlbench.a2.data import make_loader\n    from dlbench.a2.models import build_model, count_parameters\n    from dlbench.a2.losses import SegmentationLoss\n    from dlbench.common.reproducibility import collect_environment\n    if steps < 2:\n        raise ValueError("Pilot needs at least two optimization steps")\n    if dataset.split != "train":\n        raise ValueError("Optimize the pilot on train only")\n    if require_cuda and (device != "cuda" or not torch.cuda.is_available()):\n        raise RuntimeError("Select a Colab GPU runtime before the pilot/main run")\n    if config["protocol"].get("cuda_determinism") != CUDA_POLICY:\n        raise ValueError("Pilot config must declare the CUDA warning policy")\n    seed_for_unet(config["run"]["seed"])\n    loader = make_loader(dataset, batch_size=config["budget"]["batch_size"], workers=0)\n    batch = next(iter(loader))\n    images, targets = batch["image"].to(device), batch["mask"].to(device)\n    if not (targets != 255).any():\n        raise ValueError("Pilot batch contains no supervised pixels")\n    model = build_model(config["model"]).to(device).train()\n    optimizer = torch.optim.AdamW(model.parameters(), lr=config["training"]["lr"],\n                                 weight_decay=config["training"]["weight_decay"])\n    criterion = SegmentationLoss(**config["loss"])\n    if device == "cuda":\n        torch.cuda.reset_peak_memory_stats()\n    losses, durations = [], []\n    for step in range(steps):\n        if device == "cuda":\n            torch.cuda.synchronize()\n        started = time.perf_counter()\n        optimizer.zero_grad(set_to_none=True)\n        logits = model(images)\n        if logits.shape != (images.shape[0], 17, *targets.shape[1:]):\n            raise ValueError("Pilot logits contract mismatch")\n        loss = criterion(logits, targets)\n        if not torch.isfinite(loss):\n            raise FloatingPointError("Pilot loss is non-finite")\n        loss.backward()\n        if any(p.grad is not None and not torch.isfinite(p.grad).all() for p in model.parameters()):\n            raise FloatingPointError("Pilot gradient is non-finite")\n        optimizer.step()\n        if device == "cuda":\n            torch.cuda.synchronize()\n        durations.append(time.perf_counter() - started)\n        losses.append(float(loss.detach()))\n        if step in (0, steps - 1) or (step + 1) % 5 == 0:\n            print(f"Pilot {step + 1}/{steps}: loss={losses[-1]:.4f}", flush=True)\n    total = torch.cuda.get_device_properties(0).total_memory if device == "cuda" else None\n    reserved = torch.cuda.max_memory_reserved() if device == "cuda" else None\n    passed = losses[-1] < losses[0] and (reserved is None or reserved < 0.9 * total)\n    result = {"kind": "real_batch_gpu_pilot" if device == "cuda" else "cpu_development_check",\n              "device": device, "passed": passed, "config_sha256": digest(config),\n              "code_revision": CODE_REVISION, "steps": steps,\n              "cuda_determinism": deepcopy(CUDA_POLICY),\n              "batch_size": int(images.shape[0]), "canvas": list(targets.shape[-2:]),\n              "image_ids": [int(v) for v in batch["image_id"]],\n              "losses": losses, "train_step_seconds_excluding_io": durations[2:],\n              "peak_allocated_bytes": torch.cuda.max_memory_allocated() if device == "cuda" else None,\n              "peak_reserved_bytes": reserved, "gpu_total_bytes": total,\n              "environment": collect_environment(), "parameter_counts": count_parameters(model),\n              "note": "Fixed-batch loss decrease and capacity check, not full-data overfit or benchmark quality"}\n    del model, optimizer, criterion, batch, images, targets, logits, loss\n    gc.collect()\n    if device == "cuda":\n        torch.cuda.empty_cache()\n    return result\n\n\ndef freeze_config(base, pilot_record, *, approval=APPROVAL):\n    from dlbench.a2.config import validate_config\n    if not approval:\n        raise ValueError("Record instructor approval before main")\n    if base["protocol"].get("cuda_determinism") != CUDA_POLICY:\n        raise ValueError("Missing CUDA determinism policy")\n    if (not pilot_record["passed"] or pilot_record["device"] != "cuda"\n            or pilot_record["code_revision"] != CODE_REVISION\n            or pilot_record.get("cuda_determinism") != CUDA_POLICY\n            or pilot_record["config_sha256"] != digest(base)\n            or pilot_record["batch_size"] != base["budget"]["batch_size"]):\n        raise ValueError("Matching successful GPU pilot required before main")\n    result = deepcopy(base)\n    result["protocol"].update(status="frozen", approval_reference=approval,\n                              pilot_reference="sha256:" + digest(pilot_record))\n    validate_config(result, main=True)\n    return result\n\n\ndef pin_setup(path, base, pilot_record, helper_sha):\n    """Store settings once; reject changed options rather than silently overwrite."""\n    path = Path(path)\n    if path.exists():\n        saved = json.loads(path.read_text(encoding="utf-8"))\n        if digest(saved["payload"]) != saved["sha256"]:\n            raise ValueError("Setup ledger hash mismatch")\n        value = saved["payload"]\n        if value["base_sha256"] != digest(base) or value["helper_sha256"] != helper_sha:\n            raise ValueError("Settings/helper changed; use a new RUN_TAG, preserve the existing run")\n        if value["config"] != freeze_config(base, value["pilot"]):\n            raise ValueError("Frozen config does not match pilot/approval")\n        return value\n    if pilot_record is None:\n        raise ValueError("A fresh setup needs its GPU pilot")\n    value = {"created_utc": datetime.now(timezone.utc).isoformat(), "code_revision": CODE_REVISION,\n             "base_sha256": digest(base), "helper_sha256": helper_sha,\n             "pilot": pilot_record, "config": freeze_config(base, pilot_record)}\n    write_json(path, {"payload": value, "sha256": digest(value)})\n    return value\n\n\ndef install_progress(run_root):\n    """Decorate the shared engine for visibility; core checkpoint/training is unchanged."""\n    import dlbench.a2.trainer as trainer\n    from dlbench.a2.engine import run_epoch\n    from tqdm.auto import tqdm\n    journal = Path(run_root) / "epoch_progress.jsonl"\n    def visible_epoch(model, loader, criterion, device, optimizer=None):\n        phase = "train" if optimizer is not None else "validation"\n        started = time.perf_counter()\n        result = run_epoch(model, tqdm(loader, desc=phase), criterion, device, optimizer)\n        elapsed = time.perf_counter() - started\n        print(f"{phase}: loss={result[\'loss\']:.4f}, fg mIoU={result[\'foreground_miou\']}, seconds={elapsed:.1f}", flush=True)\n        with journal.open("a", encoding="utf-8") as stream:\n            stream.write(json.dumps({"phase": phase, "seconds": elapsed, "loss": result["loss"],\n                                     "foreground_miou": result["foreground_miou"]}) + "\\n")\n        return result\n    trainer.run_epoch = visible_epoch\n\n\ndef train_or_resume(config, train, validation, output, *, device="cuda"):\n    from dlbench.a2.trainer import fit\n    output = Path(output)\n    resume = output.exists()\n    if resume and not all((output / name).is_file() for name in ("metadata.json", "last.pt")):\n        raise ValueError("Run directory has no complete epoch checkpoint; preserve it and use a new RUN_TAG")\n    install_runtime_policy(config)\n    # The core trainer enforces exact config/source/device/environment identity.\n    return fit(config, train, validation, output, device=device, workers=0, resume=resume)\n', 'a2_resume': '"""Exact mismatch diagnosis and provenance-preserving host-only continuation.\n\nOriginal run/checkpoint identities are never edited. The pinned trainer still\nuses its strict identity check on every resumed run, including derived copies.\n"""\nfrom copy import deepcopy\nfrom datetime import datetime, timezone\nfrom pathlib import Path\nimport hashlib\nimport json\nimport shutil\nimport uuid\n\nHOST_ONLY_FIELDS = frozenset({\n    "environment.os_release", "environment.processor",\n    "environment.system_ram.total_bytes", "environment.system_ram.total_gb",\n})\n\n\ndef digest(value):\n    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":"),\n                                     ensure_ascii=False, allow_nan=False).encode()).hexdigest()\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef write_json(path, value):\n    path = Path(path)\n    temporary = path.with_name(path.name + ".tmp-" + uuid.uuid4().hex)\n    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False) + "\\n",\n                         encoding="utf-8")\n    temporary.replace(path)\n\n\ndef differences(previous, current, prefix=""):\n    result = []\n    if isinstance(previous, dict) and isinstance(current, dict):\n        for key in sorted(set(previous) | set(current)):\n            path = f"{prefix}.{key}" if prefix else key\n            if key not in previous or key not in current:\n                result.append({"field": path, "saved": previous.get(key, "<missing>"),\n                               "current": current.get(key, "<missing>")})\n            else:\n                result.extend(differences(previous[key], current[key], path))\n    elif type(previous) is not type(current) or previous != current:\n        result.append({"field": prefix, "saved": previous, "current": current})\n    return result\n\n\ndef mismatch_context(error):\n    """Read the actual failed fit frame; never approximate its fingerprint."""\n    trace = error.__traceback__\n    while trace:\n        values = trace.tb_frame.f_locals\n        if (trace.tb_frame.f_code.co_name == "fit"\n                and {"identity", "metadata", "fingerprint", "output"} <= set(values.keys())):\n            previous = deepcopy(values["metadata"])\n            current = deepcopy(values["identity"])\n            fingerprint = values["fingerprint"]\n            if digest(previous["identity"]) != previous["identity_sha256"]:\n                raise ValueError("Original metadata identity hash is invalid")\n            if digest(current) != fingerprint:\n                raise ValueError("Current fit identity hash is invalid")\n            return {"metadata": previous, "identity": current, "fingerprint": fingerprint,\n                    "run": Path(values["output"]).resolve(),\n                    "source": deepcopy(values.get("source", previous["source"]))}\n        trace = trace.tb_next\n    raise ValueError("Cannot find the exact failed fit identity frame; rerun the original failing cell")\n\n\ndef origin_root(root):\n    root = Path(root).resolve()\n    provenance = root / "continuation.json"\n    if not provenance.is_file():\n        return root\n    value = json.loads(provenance.read_text(encoding="utf-8"))\n    origin = Path(value["origin_root"]).resolve()\n    if not root.is_relative_to(origin / "continuations"):\n        raise ValueError("Continuation origin path is invalid")\n    return origin\n\n\ndef resolve_target(origin):\n    origin = Path(origin).resolve()\n    pointer = origin / "resume_pointer.json"\n    if not pointer.is_file():\n        return origin\n    value = json.loads(pointer.read_text(encoding="utf-8"))\n    target = (origin / value["target_relative"]).resolve()\n    if not target.is_relative_to(origin / "continuations"):\n        raise ValueError("Resume pointer escapes this run\'s continuation directory")\n    metadata = json.loads((target / "run/metadata.json").read_text(encoding="utf-8"))\n    if digest(metadata["identity"]) != metadata["identity_sha256"]:\n        raise ValueError("Continuation metadata identity hash is invalid")\n    if metadata["identity_sha256"] != value["identity_sha256"]:\n        raise ValueError("Resume pointer identity mismatch")\n    if not (target / "run/last.pt").is_file():\n        raise ValueError("Continuation pointer has no epoch checkpoint")\n    return target\n\n\ndef fork_host_continuation(context, origin, changes):\n    import torch\n    origin = Path(origin).resolve()\n    source_run = context["run"]\n    if not source_run.is_relative_to(origin) or source_run.name != "run":\n        raise ValueError("Source run is outside the selected experiment")\n    if not changes or any(item["field"] not in HOST_ONLY_FIELDS for item in changes):\n        raise ValueError("Continuation permits only RAM/kernel/processor descriptor changes")\n    old = context["metadata"]\n    current = context["identity"]\n    fingerprint = context["fingerprint"]\n    actual = differences(old["identity"], current)\n    if changes != actual:\n        raise ValueError("Mismatch report does not match the identities")\n    checkpoint = torch.load(source_run / "last.pt", map_location="cpu", weights_only=False)\n    if checkpoint["identity_sha256"] != old["identity_sha256"]:\n        raise ValueError("Original checkpoint identity mismatch")\n    if checkpoint["epoch"] < 1 or len(checkpoint["history"]) != checkpoint["epoch"]:\n        raise ValueError("Original epoch/history is inconsistent")\n    best = checkpoint["best"]\n    if best is None:\n        raise ValueError("Original checkpoint has no selected validation checkpoint")\n    # Recover the best copy from the authoritative last state if its write was\n    # interrupted after last.pt was saved. Never write into the parent run.\n    if best["epoch"] == checkpoint["epoch"]:\n        best_checkpoint = checkpoint\n    else:\n        best_checkpoint = torch.load(source_run / "best.pt", map_location="cpu", weights_only=False)\n        if (best_checkpoint["identity_sha256"] != old["identity_sha256"]\n                or best_checkpoint["epoch"] != best["epoch"]):\n            raise ValueError("Original best checkpoint disagrees with last.pt")\n    inputs = {name: file_hash(source_run / name) for name in\n              ("metadata.json", "last.pt", "best.pt", "effective_config.json")\n              if (source_run / name).is_file()}\n    if json.loads((source_run / "effective_config.json").read_text(encoding="utf-8")) != current["config"]:\n        raise ValueError("Effective config does not match the checkpoint")\n    suffix = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]\n    target = origin / "continuations" / suffix\n    run = target / "run"\n    run.mkdir(parents=True, exist_ok=False)\n    for name in ("setup.json", "frozen_config.json", "pilot.json", "manifest.json",\n                 "data_checks.json", "data_overlay.png", "helper_snapshot.py"):\n        path = origin / name\n        if path.is_file():\n            shutil.copy2(path, target / name)\n    if (source_run / "source").is_dir():\n        shutil.copytree(source_run / "source", run / "source")\n    record = {"kind": "same_experiment_host_continuation", "origin_root": str(origin),\n              "parent_run": str(source_run), "parent_run_id": old.get("run_id"),\n              "parent_identity_sha256": old["identity_sha256"],\n              "current_identity_sha256": fingerprint, "differences": changes,\n              "resumed_epoch": checkpoint["epoch"], "original_input_sha256": inputs,\n              "created_utc": datetime.now(timezone.utc).isoformat(),\n              "recovery_source_sha256": file_hash(Path(__file__)),\n              "note": "Same config/source/data/GPU/libraries; host descriptors differ. Not a new seed or independent run; no bitwise GPU claim."}\n    metadata = deepcopy(old)\n    metadata.update(run_id=str(uuid.uuid4()), identity=current, identity_sha256=fingerprint,\n                    environment=current["environment"], source=context["source"], continuation=record)\n    metadata["experiment_id"] = old.get("experiment_id", old.get("run_id"))\n    write_json(run / "metadata.json", metadata)\n    write_json(run / "effective_config.json", current["config"])\n    for name, state in (("last.pt", checkpoint), ("best.pt", best_checkpoint)):\n        copied = dict(state)\n        copied["identity_sha256"] = fingerprint  # Only the derived copy changes identity.\n        torch.save(copied, run / name)\n    write_json(run / "history.json", checkpoint["history"])\n    write_json(run / "metrics.json", {"best": best, "epochs_completed": checkpoint["epoch"],\n                                      "kind": metadata["kind"]})\n    write_json(target / "continuation.json", record)\n    shutil.copy2(Path(__file__), target / "resume_recovery_snapshot.py")\n    for name, previous_hash in inputs.items():\n        if file_hash(source_run / name) != previous_hash:\n            raise ValueError("Original input changed during copying; continuation not published")\n    # Publish a routing pointer only after all derived checkpoint files exist.\n    write_json(origin / "resume_pointer.json", {"target_relative": target.relative_to(origin).as_posix(),\n               "identity_sha256": fingerprint})\n    return target\n\n\ndef train_with_recovery(config, train, validation, run_root, setup, *, runner=None):\n    """Keep strict resume; branch only when exact drift is entirely descriptive."""\n    root = origin_root(run_root)\n    target = resolve_target(root)\n    invoke = runner or setup.train_or_resume\n    setup.install_progress(target)\n    try:\n        return target, invoke(config, train, validation, target / "run")\n    except ValueError as error:\n        if str(error) != "Resume config/source/device identity mismatch":\n            raise\n        context = mismatch_context(error)\n        changes = differences(context["metadata"]["identity"], context["identity"])\n        report = {"differences": changes, "host_only": bool(changes) and all(\n                  item["field"] in HOST_ONLY_FIELDS for item in changes)}\n        write_json(root / "resume_diagnostic.json", report)\n        print(json.dumps(report, ensure_ascii=False, indent=2), flush=True)\n        if not report["host_only"]:\n            raise ValueError("Resume stopped: numerical/config/source/data identity differs. Send resume_diagnostic.json; all checkpoints preserved.") from error\n        target = fork_host_continuation(context, root, changes)\n        print("Continuing the same experiment from the saved epoch in:", target, flush=True)\n        setup.install_progress(target)\n        return target, invoke(config, train, validation, target / "run")\n\n\ndef launch(namespace):\n    required = ("setup", "CONFIG", "train_data", "validation_data", "RUN_ROOT")\n    if any(name not in namespace for name in required):\n        raise ValueError("Run notebook setup/data/pilot cells first, then this recovery cell")\n    if not namespace.get("RUN_MAIN", False):\n        raise ValueError("RUN_MAIN=False; enable it before resuming training")\n    target, result = train_with_recovery(namespace["CONFIG"], namespace["train_data"],\n                    namespace["validation_data"], namespace["RUN_ROOT"], namespace["setup"])\n    namespace["RUN_ROOT"] = target\n    namespace["outcome"] = result\n    namespace["setup"].write_json(target / "outcome.json", result)\n    print(result)\n    print("Then run the original notebook\'s best-validation/curves/ZIP cell.")\n', 'a2_test': '"""Evaluate fixed test data once after main completion; never retrain/select on test."""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\n\n\ndef file_hash(path):\n    result = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1024 * 1024), b""):\n            result.update(chunk)\n    return result.hexdigest()\n\n\ndef selected_run(root):\n    root = Path(root).resolve()\n    pointer = root / "resume_pointer.json"\n    if pointer.is_file():\n        value = json.loads(pointer.read_text(encoding="utf-8"))\n        target = (root / value["target_relative"]).resolve()\n        if not target.is_relative_to(root / "continuations"):\n            raise ValueError("Resume pointer escapes the experiment")\n        metadata = json.loads((target / "run/metadata.json").read_text(encoding="utf-8"))\n        if metadata["identity_sha256"] != value["identity_sha256"]:\n            raise ValueError("Resume pointer identity mismatch")\n        return target\n    return root\n\n\ndef assert_complete(state, config, metadata):\n    from dlbench.a2.data import content_hash\n    from dlbench.a2.config import validate_config\n    validate_config(config, main=True)\n    if metadata["kind"] != "main" or metadata["identity"]["development"]:\n        raise ValueError("Test requires a main checkpoint")\n    if metadata["identity"]["config"] != config:\n        raise ValueError("Saved effective config differs from run identity")\n    if content_hash(metadata["identity"]) != metadata["identity_sha256"]:\n        raise ValueError("Metadata identity hash mismatch")\n    if state["identity_sha256"] != metadata["identity_sha256"]:\n        raise ValueError("Last checkpoint identity mismatch")\n    epoch = state["epoch"]\n    if type(epoch) is not int or not 1 <= epoch <= config["budget"]["epochs"]:\n        raise ValueError("Invalid completed epoch")\n    if len(state["history"]) != epoch or state["best"] is None:\n        raise ValueError("Incomplete checkpoint history/selection")\n    completed = epoch == config["budget"]["epochs"]\n    early_stopped = state["bad_epochs"] >= config["budget"]["patience"]\n    if not (completed or early_stopped):\n        raise ValueError(f"Main run is only at epoch {epoch}; resume training before test evaluation")\n    return "budget_completed" if completed else "early_stopping"\n\n\ndef run_test(root, data_root, manifest, *, device="cuda"):\n    import torch\n    from dlbench.a2.data import TrashCanDataset, content_hash\n    from dlbench.a2.trainer import evaluate_checkpoint, save_json, source_identity\n    root = selected_run(root)\n    run = root / "run"\n    config = json.loads((run / "effective_config.json").read_text(encoding="utf-8"))\n    metadata = json.loads((run / "metadata.json").read_text(encoding="utf-8"))\n    state = torch.load(run / "last.pt", map_location="cpu", weights_only=False)\n    stop_reason = assert_complete(state, config, metadata)\n    selected_epoch = state["best"]["epoch"]\n    best_state = torch.load(run / "best.pt", map_location="cpu", weights_only=False)\n    if (best_state["epoch"] != selected_epoch\n            or best_state["identity_sha256"] != metadata["identity_sha256"]):\n        raise ValueError("best.pt differs from validation selection; repair checkpoint before test")\n    del best_state\n    if source_identity()["source_digest"] != metadata["identity"]["source_digest"]:\n        raise ValueError("Current model/evaluator source differs from the run snapshot")\n    del state\n    dataset = TrashCanDataset(data_root, manifest, "test", config["preprocessing"],\n                              seed=config["run"]["seed"], color_jitter=False)\n    if len(dataset) != 1078:\n        raise ValueError("Expected all 1078 fixed test images")\n    checkpoint_sha = file_hash(run / "best.pt")\n    membership = content_hash(dataset.rows)\n    identity = {"checkpoint_sha256": checkpoint_sha,\n                "run_identity_sha256": metadata["identity_sha256"], "membership_sha256": membership}\n    output = root / "test_best.json"\n    marker = root / "test_complete.json"\n    if output.is_file() and marker.is_file():\n        record = json.loads(marker.read_text(encoding="utf-8"))\n        if record["identity"] == identity and record["output_sha256"] == file_hash(output):\n            result = json.loads(output.read_text(encoding="utf-8"))\n            print("Reusing matching verified test result; no inference or training repeated.")\n        else:\n            raise ValueError("Existing test result/cache identity changed; preserve it for review")\n    else:\n        # Core evaluator loads selected best.pt; no test score affects selection.\n        result = evaluate_checkpoint(run, dataset, device=device, workers=0)\n        result["training_stop_reason"] = stop_reason\n        result["parameter_counts"] = metadata["parameter_counts"]\n        from dlbench.common.reproducibility import collect_environment\n        result["evaluation_environment"] = collect_environment()\n        result["evaluation_device"] = str(device)\n        if any(result[key] != value for key, value in identity.items()):\n            raise ValueError("Test evaluator provenance differs from selected checkpoint")\n        save_json(output, result)\n        save_json(marker, {"identity": identity, "output_sha256": file_hash(output)})\n    if result["split"] != "test":\n        raise ValueError("Cached output is not the test partition")\n    if result["selected_epoch"] != selected_epoch:\n        raise ValueError("Test output selected epoch differs from validation selection")\n    metrics = result["metrics"]\n    with (root / "test_per_class.csv").open("w", encoding="utf-8", newline="") as stream:\n        fields = ("id", "name", "iou", "dice", "support", "predicted")\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader()\n        writer.writerows(metrics["per_class"])\n    with (root / "test_confusion.csv").open("w", encoding="utf-8", newline="") as stream:\n        csv.writer(stream).writerows(metrics["confusion_matrix"])\n    print("TEST foreground mIoU:", metrics["foreground_miou"])\n    print("TEST foreground Dice:", metrics["foreground_dice"])\n    print("TEST trash-only mIoU:", metrics["trash_miou"])\n    print("Saved:", output)\n    return root, result\n\n\ndef launch(namespace):\n    required = ("RUN_ROOT", "DATA_ROOT", "MANIFEST")\n    if any(key not in namespace for key in required):\n        raise ValueError("Run notebook setup/data cells first")\n    import torch\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    root, result = run_test(namespace["RUN_ROOT"], namespace["DATA_ROOT"], namespace["MANIFEST"], device=device)\n    namespace["RUN_ROOT"], namespace["best_test"] = root, result\n', 'a2_segformer': '"""Paired Colab orchestration; uses the pinned shared A2 trainer unchanged."""\nfrom copy import deepcopy\nfrom pathlib import Path\nimport csv\nimport gc\nimport importlib.metadata\nimport json\nimport os\nimport zipfile\n\nimport a2_support as support\nimport a2_resume as recovery\nimport a2_test as testing\n\nRUN_NAMES = ("S0", "S1")\nEXPECTED_PARAMS = {"total_params": 3718513, "trainable_params": 3718513, "frozen_params": 0}\n\n\ndef paired_runtime():\n    """Keep a fresh S1 run on the same numerical stack as an earlier S0."""\n    import torch\n    from dlbench.common.reproducibility import collect_environment\n    support.seed_for_unet(69420)\n    environment = deepcopy(collect_environment())\n    for field in recovery.HOST_ONLY_FIELDS:\n        parts = field.split(\'.\')[1:]\n        parent = environment\n        for part in parts[:-1]:\n            parent = parent.get(part, {})\n        parent.pop(parts[-1], None)\n    dependencies = {name: importlib.metadata.version(name) for name in\n                    (\'Pillow\', \'transformers\', \'huggingface-hub\', \'tqdm\')}\n    return {"environment": environment, "dependencies": dependencies,\n            "cpu_threads": torch.get_num_threads(), "default_dtype": str(torch.get_default_dtype())}\n\n\ndef prepare_bases(repo, *, batch_size=4, epochs=40, patience=8, lr=6e-5):\n    from dlbench.a2.config import load_config, validate_config\n    result = {}\n    for name, file in zip(RUN_NAMES, ("segformer_off.py", "segformer_on.py")):\n        config = load_config(Path(repo) / "configs/a2" / file)\n        config["protocol"]["cuda_determinism"] = deepcopy(support.CUDA_POLICY)\n        config["budget"].update(batch_size=batch_size, epochs=epochs, patience=patience)\n        config["training"]["lr"] = lr\n        validate_config(config)\n        result[name] = config\n    assert_paired(result)\n    return result\n\n\ndef assert_paired(configs):\n    """Run name and train color jitter are the sole experimental differences."""\n    comparable = []\n    for name in RUN_NAMES:\n        config = deepcopy(configs[name])\n        if config["model"]["name"] != "segformer":\n            raise ValueError("Expected pretrained SegFormer in both arms")\n        if config["run"] != {"name": name, "seed": 69420, "color_jitter": name == "S1"}:\n            raise ValueError("Expected paired S0/S1 at seed69420")\n        config.pop("run")\n        # References record the separate pilot evidence, not a treatment change.\n        for key in ("pilot_reference",):\n            config["protocol"].pop(key, None)\n        comparable.append(config)\n    if comparable[0] != comparable[1]:\n        raise ValueError("S0/S1 settings differ beyond color augmentation")\n    return support.digest(comparable[0])\n\n\ndef make_datasets(root, manifest, config):\n    from dlbench.a2.data import TrashCanDataset\n    # Trainer checks the flag for both adapters. The adapter applies it only\n    # when split == \'train\', so validation images are never color-jittered.\n    args = dict(seed=config["run"]["seed"], color_jitter=config["run"]["color_jitter"])\n    return tuple(TrashCanDataset(root, manifest, split, config["preprocessing"], **args)\n                 for split in ("train", "validation"))\n\n\ndef pin_pair(pair_root, bases, bundle_sha):\n    """Lock both treatment configs before any main training or test evaluation."""\n    pair_root = Path(pair_root)\n    expected = {"bases": bases, "bundle_sha256": bundle_sha,\n                "paired_settings_sha256": assert_paired(bases), "code_revision": support.CODE_REVISION,\n                "numerical_runtime": paired_runtime()}\n    path = pair_root / "pair_setup.json"\n    if path.exists():\n        value = json.loads(path.read_text(encoding="utf-8"))\n        if value != expected:\n            raise ValueError("Pair settings/helper/runtime changed; preserve this pair and restore matching runtime/settings or use a new PAIR_TAG")\n    else:\n        pair_root.mkdir(parents=True, exist_ok=True)\n        if any(pair_root.glob("S*/run/metadata.json")):\n            raise ValueError("Existing run without pair ledger; preserve it for review")\n        support.write_json(path, expected)\n    return expected\n\n\ndef pin_run(root, base, train, bundle_sha, *, pilot_steps=20):\n    from dlbench.common.reproducibility import collect_environment\n    root = Path(root)\n    root.mkdir(parents=True, exist_ok=True)\n    ledger = root / "setup.json"\n    if ledger.exists():\n        locked = support.pin_setup(ledger, base, None, bundle_sha)\n    else:\n        pilot = support.pilot(base, train, steps=pilot_steps)\n        support.write_json(root / "pilot.json", pilot)\n        if not pilot["passed"] or pilot["parameter_counts"] != EXPECTED_PARAMS:\n            raise ValueError("SegFormer pilot failed; inspect loss/VRAM/counts before main")\n        locked = support.pin_setup(ledger, base, pilot, bundle_sha)\n    if locked["pilot"]["parameter_counts"] != EXPECTED_PARAMS:\n        raise ValueError("Unexpected SegFormer total/trainable/frozen parameters")\n    if locked["pilot"]["environment"]["gpus"] != collect_environment()["gpus"]:\n        raise ValueError("GPU differs from pinned pilot; retain this run for diagnosis")\n    support.write_json(root / "frozen_config.json", locked["config"])\n    return locked["config"]\n\n\ndef check_initial_pair(pair_root):\n    """Reject unequal initial pretrained/random head states for paired runs."""\n    records = []\n    for name in RUN_NAMES:\n        root = recovery.resolve_target(Path(pair_root) / name)\n        path = root / "run/metadata.json"\n        if path.is_file():\n            records.append(json.loads(path.read_text(encoding="utf-8")))\n    if len(records) == 2:\n        if records[0]["initial_state_sha256"] != records[1]["initial_state_sha256"]:\n            raise ValueError("Paired initial model weights differ; preserve outputs for review")\n        if records[0]["parameter_counts"] != records[1]["parameter_counts"]:\n            raise ValueError("Paired model parameter counts differ")\n    return len(records) == 2\n\n\ndef train_run(config, train, validation, root):\n    target, result = recovery.train_with_recovery(config, train, validation, root, support)\n    support.write_json(target / "outcome.json", result)\n    check_initial_pair(Path(root).parent)\n    return target, result\n\n\ndef validation_replay(root, validation, *, device="cuda"):\n    from dlbench.a2.trainer import evaluate_checkpoint\n    root = Path(root)\n    metadata = json.loads((root / "run/metadata.json").read_text(encoding="utf-8"))\n    checkpoint_sha = support.sha256_file(root / "run/best.pt")\n    output = root / "validation_best.json"\n    if output.exists():\n        value = json.loads(output.read_text(encoding="utf-8"))\n        if (value["checkpoint_sha256"] == checkpoint_sha\n                and value["run_identity_sha256"] == metadata["identity_sha256"]):\n            return value\n    value = evaluate_checkpoint(root / "run", validation, device=device, workers=0)\n    support.write_json(output, value)\n    return value\n\n\ndef export_run(origin, active, *, download=False):\n    import matplotlib.pyplot as plt\n    origin, active = Path(origin), Path(active)\n    history = json.loads((active / "run/history.json").read_text(encoding="utf-8"))\n    fig, axes = plt.subplots(1, 2, figsize=(11, 4))\n    for split in ("train", "validation"):\n        for ax, key in zip(axes, ("loss", "foreground_miou")):\n            ax.plot([r["epoch"] for r in history], [r[split][key] for r in history], label=split)\n    for ax, label in zip(axes, ("CE + Dice loss", "Foreground mIoU")):\n        ax.set(xlabel="Epoch", ylabel=label); ax.legend(); ax.grid(alpha=.2)\n    fig.tight_layout(); fig.savefig(active / "learning_curves.png", dpi=140)\n    plt.show(); plt.close(fig)\n    package = origin.parent / (origin.name + "_results.zip")\n    temporary = package.with_suffix(".zip.partial")\n    # Include origin ledger, routing pointer and all parents/continuations.\n    # This archive is sufficient for restoring the experiment on a fresh runtime.\n    with zipfile.ZipFile(temporary, "w", compression=zipfile.ZIP_DEFLATED) as archive:\n        for path in sorted(origin.rglob("*")):\n            if path.is_file():\n                archive.write(path, path.relative_to(origin.parent))\n        archive.write(origin.parent / "pair_setup.json", "pair_setup.json")\n    os.replace(temporary, package)\n    print("Saved:", package, flush=True)\n    if download:\n        from google.colab import files\n        files.download(str(package))\n    gc.collect()\n    import torch\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n    return package\n\n\ndef run_selected(pair_root, selected, configs, datasets, data_root, manifest, *,\n                 run_main=True, run_test=True, download=False):\n    selected = tuple(selected)\n    if not selected or len(set(selected)) != len(selected) or any(n not in RUN_NAMES for n in selected):\n        raise ValueError("RUNS must select S0 and/or S1 without duplicates")\n    assert_paired(configs)\n    ledger = Path(pair_root) / \'pair_setup.json\'\n    if ledger.is_file():\n        if json.loads(ledger.read_text(encoding=\'utf-8\'))[\'numerical_runtime\'] != paired_runtime():\n            raise ValueError(\'Numerical runtime changed after paired setup; restore the matching environment\')\n    if not run_main:\n        print("RUN_MAIN=False: data and paired pilots only; no main/test.")\n        return {}\n    results = {}\n    for name in selected:\n        origin = Path(pair_root) / name\n        print(f"===== {name}: start/resume =====", flush=True)\n        train, validation = datasets[name]\n        active, outcome = train_run(configs[name], train, validation, origin)\n        # Test helper verifies the main completion/early-stop gate before test.\n        validation_replay(active, validation)\n        if run_test:\n            testing.run_test(active, data_root, manifest, device="cuda")\n        export_run(origin, active, download=download)\n        results[name] = {"active_root": str(active), "outcome": outcome}\n    write_summary(pair_root)\n    return results\n\n\ndef write_summary(pair_root):\n    rows = []\n    for name in RUN_NAMES:\n        root = recovery.resolve_target(Path(pair_root) / name)\n        path = root / "test_best.json"\n        if path.exists():\n            value = json.loads(path.read_text(encoding="utf-8"))\n            rows.append({"run": name, "selected_epoch": value["selected_epoch"],\n                         **{k: value["metrics"][k] for k in\n                            ("foreground_miou", "foreground_dice", "trash_miou")}})\n    if rows:\n        support.write_json(Path(pair_root) / "test_summary.json", rows)\n        with (Path(pair_root) / "test_summary.csv").open("w", encoding="utf-8", newline="") as stream:\n            writer = csv.DictWriter(stream, fieldnames=list(rows[0]))\n            writer.writeheader(); writer.writerows(rows)\n        print(json.dumps(rows, indent=2))\n    return rows\n'}
HELPER_HASHES = {'a2_support': 'd628fde2c2bee63a37b3c50e71681b8501974f685ba7f56d6135d2004385a4a4', 'a2_resume': 'a98e3a0f8ec4bdb6dab1d867203ee142997fe6ca6e6ba5458aed345f3b216644', 'a2_test': 'fdbf23dd6686ffbe6e9fdfc51a637f4b70fcd68c041aae5f033b2f6ae4ddc482', 'a2_segformer': '0d52f285d03f87668957a034af73eb339298e64895f15f520d105da3fef42212'}
BUNDLE_SHA = '6eb171d43ffee55b8f252c2d307576855f07e10e8cd60c0a678d0bda7d0b017d'
assert hashlib.sha256(json.dumps(HELPER_HASHES, sort_keys=True, separators=(',', ':')).encode()).hexdigest() == BUNDLE_SHA
for name, source in HELPERS.items():
    assert hashlib.sha256(source.encode()).hexdigest() == HELPER_HASHES[name]
    path = Path('/content') / (name + '.py')
    path.write_text(source, encoding='utf-8')
    spec = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
import a2_support as support
import a2_segformer as setup
MANIFEST = Path('/content/trashcan_material_seed36_v1.json')
MANIFEST.write_bytes(zlib.decompress(base64.b64decode('...')))
from dlbench.a2.data import read_manifest
manifest = read_manifest(MANIFEST)
assert len(manifest['images']) == 7212
print('Manifest:', manifest['content_sha256'])


## 4. Dataset và pin cặp cấu hình

Tái dùng `MyDrive/dlbench_a2_cache/dataset.zip` của U-Net. Nếu cache chưa có, tải lại từ Drive nhóm;
SHA-256 archive phải khớp EDA. Ảnh được giải nén vào SSD `/content`.
S0/S1 khác duy nhất tên run và bật/tắt màu train; cùng split, seed, normalization và budget.
Validation/test luôn giữ nguyên màu. Không tự giảm batch khi OOM: đổi settings và PAIR_TAG cho **cả cặp**.


In [ ]:
BASES = setup.prepare_bases(REPO, batch_size=BATCH_SIZE, epochs=MAX_EPOCHS,
                            patience=PATIENCE, lr=LEARNING_RATE)
setup.pin_pair(PAIR_ROOT, BASES, BUNDLE_SHA)
ARCHIVE = support.acquire_archive(CACHE / 'dataset.zip', support.download_from_drive)
DATA_ROOT = support.extract_material(ARCHIVE, Path('/content/dlbench_a2_material'))
DATASETS = {name: setup.make_datasets(DATA_ROOT, MANIFEST, base) for name, base in BASES.items()}
assert all((len(train), len(val)) == (5057, 1077) for train, val in DATASETS.values())
if VERIFY_ALL_MASKS:
    verification = support.verify_training_data(*DATASETS['S0'])
else:
    for ds in DATASETS['S0']:
        for index in range(min(4, len(ds))):
            ds[index]
    verification = {'checked': 'first four train/validation only', 'test_evaluated': False}
support.write_json(PAIR_ROOT / 'data_checks.json', verification)
for name in setup.RUN_NAMES:
    root = PAIR_ROOT / name
    root.mkdir(parents=True, exist_ok=True)
    (root / 'manifest.json').write_bytes(MANIFEST.read_bytes())
    support.write_json(root / 'data_checks.json', verification)
    support.write_json(root / 'helper_bundle.json', {'sha256': BUNDLE_SHA, 'files': HELPER_HASHES})
    (root / 'helper_snapshot.py').write_text(HELPERS['a2_segformer'], encoding='utf-8')
    for module_name, source in HELPERS.items():
        (root / (module_name + '_snapshot.py')).write_text(source, encoding='utf-8')
print('Train/validation: 5057 / 1077 | test giữ riêng: 1078')


## 5. Kiểm tra ảnh-mask và augmentation

Hàng S0/S1 cùng ảnh train, cùng epoch: RGB, mask và overlay. S1 augmentation có xác suất 0.5,
nên một ảnh có thể trông giống S0. Brightness/contrast/saturation ±10%, hue ±0.01; không đổi mask.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for row, name in enumerate(setup.RUN_NAMES):
    sample = DATASETS[name][0][0]
    rgb = sample['image'].permute(1, 2, 0).numpy()
    prep = BASES[name]['preprocessing']
    rgb = np.clip(rgb * np.array(prep['std']) + np.array(prep['mean']), 0, 1)
    mask = sample['mask'].numpy()
    colors = plt.get_cmap('tab20')(np.minimum(mask, 16) / 19)[..., :3]
    colors[mask == 255] = .5
    for ax, content, label in zip(axes[row], (rgb, colors, .6*rgb+.4*colors), ('RGB', 'Target', 'Overlay')):
        ax.imshow(content); ax.set_title(f'{name}: {label}'); ax.axis('off')
fig.tight_layout()
for name in setup.RUN_NAMES:
    fig.savefig(PAIR_ROOT / name / 'data_overlay.png', dpi=140)
plt.show(); plt.close(fig)


## 6. GPU pilots và khóa cấu hình S0/S1

Mỗi arm dùng 20 bước trên một batch train thật để kiểm tra loss/gradient và headroom VRAM 10%.
Khi đã có setup ledger hợp lệ, dùng lại pilot cũ. Main khởi tạo lại pretrained/head cùng seed69420.
Thông báo classifier 150 → 17 lớp được khởi tạo mới là dự kiến. Mọi tham số đều trainable:
**3.718.513 total = trainable, frozen = 0**. Không tiếp tục nếu pilot thất bại.


In [ ]:
CONFIGS = {}
for name in setup.RUN_NAMES:
    print('Pilot/setup:', name, flush=True)
    CONFIGS[name] = setup.pin_run(PAIR_ROOT / name, BASES[name], DATASETS[name][0],
                                  BUNDLE_SHA, pilot_steps=PILOT_STEPS)
    print(name, CONFIGS[name]['budget'], CONFIGS[name]['training'])
setup.assert_paired(CONFIGS)
print('Cả hai arm đã pin trước train/test. RUNS:', RUNS)


## 7. Train → validation best → test → ZIP, lần lượt theo RUNS

Mỗi run chọn **best.pt bằng validation foreground mIoU**, tie bằng loss thấp hơn rồi epoch sớm hơn.
Test chỉ chạy sau khi đủ budget hoặc early stopping; dùng 1.078 ảnh cố định và lưu cache có checksum.
Không đổi cấu hình theo test. ZIP từng run có cả checkpoint, config, helper, manifest, lịch sử,
validation/test/per-class/confusion; thư mục cặp có test_summary.csv/json sau các run đã xong.

Nếu runtime ngắt: mở lại notebook, giữ PAIR_TAG/settings và Run all. Run hoàn tất được tiếp nhận,
test cache được tái dùng; run dở tiếp tục từ epoch hoàn tất gần nhất. Host RAM/kernel/processor drift
có thể tạo continuation giữ nguyên bản gốc; đổi GPU/thư viện/code/config/data sẽ dừng và ghi diagnostic.
Nếu muốn chạy từng run, chọn RUNS=['S0'] rồi lần sau ['S1'] với cùng PAIR_TAG và settings.


In [ ]:
RESULTS = setup.run_selected(PAIR_ROOT, RUNS, CONFIGS, DATASETS, DATA_ROOT, MANIFEST,
                             run_main=RUN_MAIN, run_test=RUN_TEST_AFTER_TRAIN,
                             download=DOWNLOAD_RESULTS)
print('Kết quả trên Drive:', PAIR_ROOT)
